# Sales Analytics Dashboard & Forecasting

This notebook analyzes company sales data to build a business dashboard and forecast future sales trends.

## Objectives
- Analyze historical sales performance
- Identify top-performing products and regions
- Monitor key sales KPIs
- Build dashboard-style visualizations
- Forecast future sales using time series models

## Tools Used
- Python
- Pandas
- Matplotlib / Seaborn
- Plotly
- Scikit-learn / Statsmodels

## Business Problem

Understanding sales trends is critical for strategic planning. This project analyzes historical sales data to identify performance patterns and predict future revenue.

Key business questions:
1. What are the overall sales trends?
2. Which products and regions generate the most revenue?
3. How do sales change over time?
4. Can we forecast future sales to support planning?

## Dataset Overview

The dataset contains transaction-level sales data. There are 3 tables

### Daily Sales Data per item

| Column | Description |
|------|-------------|
| Item | Name of item |
| Sales Rank  Quantity (Lunch / Dinner) | Rank of Quantity Sold |
| Sales Rank Amount (Lunch / Dinner) | Rank of Sales Amount |
| Sales Summary (Cash / PayNow) | Monetary Value of daily sales |


### Daily Sales Data per hour

| Column | Description |
|------|-------------|
| Time (per hour) | Starting time of the hour |
| Orders | Number of orders |
| Amount | Monetary Value of hourly sales |

### Monthly Sales Data

| Column | Description |
|------|-------------|
| Month/Year | Month and year |
| Rent | Cost of rent |
| Salary | Cost of employee salaries |
| CPF/Levy | Cost of employee CPF |
| Supplier | Cost of supplier |
| Utilities | Cost of utilities |
| Season Parking | Cost for parking |
| Total Sales | Business revenue |
| Profit/Loss | Net Profit or Loss |

Source: Koryori Hayashi

## Import Libraries

We import the required Python libraries for data processing, visualization, and forecasting.

In [43]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')


## Load Dataset

The sales dataset is loaded into a Pandas DataFrame for analysis.

In [44]:
def find_data_xlsx() -> Path:
    candidates = [
        Path('data.xlsx'),
        Path('data') / 'data.xlsx',
        Path('..') / 'data' / 'data.xlsx',  # when running from Dashboard/Calculations
        Path('Dashboard') / 'data' / 'data.xlsx',
        Path('..') / 'Dashboard' / 'data' / 'data.xlsx',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find Dashboard/data/data.xlsx. Check your working directory.')


XLSX_PATH = find_data_xlsx()
XLSX_PATH


WindowsPath('C:/Users/65967/OneDrive - Singapore Management University/Documents/GitHub/Koryori_Hayashi/Dashboard/data/data.xlsx')

In [45]:
# Read the whole sheet as a raw grid (the 3 tables live side-by-side in the same sheet)
raw = pd.read_excel(XLSX_PATH, sheet_name=0, header=None)
raw.shape


(79, 20)

In [46]:
HEADER_ROW = 4  # row 5 in Excel (0-based index)

# Table 1: Daily Sales per item (A:D)
items = raw.iloc[HEADER_ROW:, 0:4].copy()
items.columns = raw.iloc[HEADER_ROW, 0:4].tolist()
items = items.iloc[1:].reset_index(drop=True)
items = items.dropna(subset=[items.columns[0]])

# Table 2: Daily Sales per hour (F:H)
hourly = raw.iloc[HEADER_ROW:, 5:8].copy()
hourly.columns = raw.iloc[HEADER_ROW, 5:8].tolist()
hourly = hourly.iloc[1:].reset_index(drop=True)
hourly = hourly.dropna(subset=[hourly.columns[0]])

# Table 3: Monthly Sales Data (J:R)
monthly = raw.iloc[HEADER_ROW:, 9:18].copy()
monthly.columns = raw.iloc[HEADER_ROW, 9:18].tolist()
monthly = monthly.iloc[1:].reset_index(drop=True)
monthly = monthly.dropna(subset=[monthly.columns[0]])

items.head(), hourly.head(), monthly.head()


(                        Item Sales Rank  Quantity (Lunch / Dinner) Sales Rank Amount (Lunch / Dinner)  \
 0         Chicken Nanban Don                                     1                                  2   
 1         Pork Shogayaki Don                                     2                                  1   
 2        Salmon Teriyaki Don                                     3                                  3   
 3  Salmon Belly Teriyaki Don                                     4                                  4   
 4       Chicken Teriyaki Don                                     5                                  5   
 
   Sales Summary (Cash / PayNow)  
 0                           420  
 1                           450  
 2                           400  
 3                           380  
 4                           360  ,
   Time (per hour) Orders  Amount
 0        11:00:00     12  149.27
 1        12:00:00     11  138.33
 2        13:00:00     11  146.28
 3        14:00

## Feature Engineering

Additional features are created to support analysis and forecasting.

New variables include:
- Year
- Month
- Quarter
- Year-Month for time series analysis

In [47]:
# Some Excel exports label this column as 'Month / Year'. Normalize it.

monthly['Month'] = pd.to_datetime(monthly["Month / Year"], errors='coerce')
monthly = monthly.sort_values('Month').reset_index(drop=True)

monthly['Year'] = monthly['Month'].dt.year
monthly['MonthNum'] = monthly['Month'].dt.month
monthly['Quarter'] = monthly['Month'].dt.to_period('Q').astype(str)
monthly['YearMonth'] = monthly['Month'].dt.to_period('M').astype(str)

monthly[['Month', 'YearMonth', 'Year','MonthNum','Quarter','Total Sales', 'Profit / Loss']].head()


,Month,YearMonth,Year,MonthNum,Quarter,Total Sales,Profit / Loss
0,2025-01-01,2025-01,2025,1,2025Q1,3404,111.6
1,2025-02-01,2025-02,2025,2,2025Q1,2173.6,88.64
2,2025-03-01,2025-03,2025,3,2025Q1,3313.8,225.25
3,2025-04-01,2025-04,2025,4,2025Q2,1843,235.54
4,2025-05-01,2025-05,2025,5,2025Q2,1812.8,50.89


## Key Performance Indicators (KPIs)

Important metrics used to evaluate business performance:

- Total Revenue
- Total Orders
- Total Profit
- Average Order Value
- Total Units Sold

## Exploratory Data Analysis

EDA helps understand patterns, distributions, and relationships within the sales data.

### Sales Distribution

Analyze how sales values are distributed across transactions.

### Monthly Sales Trend

Visualizing sales over time helps identify seasonality and growth patterns.

## Top Performing Products

This section identifies the highest revenue-generating products.

## Profitability Analysis

Not all high-selling products generate high profit.

This section evaluates profit margins across products and categories.

## Sales Dashboard

The following visualizations summarize key insights for decision-makers.

Dashboard components include:

- Revenue trend
- Sales by category
- Regional performance
- Top products
- Profit distribution

## Preparing Data for Forecasting

To forecast sales, the dataset is aggregated into monthly revenue values.

Time series forecasting requires:
- chronological order
- evenly spaced time intervals

## Sales Forecasting

Forecasting helps predict future revenue trends based on historical data.

Methods that can be used:
- Linear Regression
- ARIMA
- Prophet
- Exponential Smoothing

## Scenario Simulation & Future Planning

Beyond forecasting future sales based on historical data, scenario simulations allow us to test how different business strategies might impact future revenue.

These simulations help answer questions such as:

- What if marketing increases sales volume?
- What if prices increase?
- What if demand drops due to market conditions?

The following scenarios model potential business outcomes.

### Scenario 1: Marketing Campaign Impact

Assumption:
- A marketing campaign increases sales volume by 10%.

Objective:
Estimate the potential revenue increase if marketing improves product visibility and demand.

Expected Outcome:
Higher monthly sales compared to the baseline forecast.

### Scenario 2: Price Increase Strategy

Assumption:
- Product prices increase by 5%.

Objective:
Estimate how a moderate price increase affects revenue.

Risk Consideration:
Price increases may reduce demand, so this scenario assumes demand remains constant.

### Scenario 3: Market Slowdown

Assumption:
- Market demand decreases by 8%.

Objective:
Evaluate how economic downturns or market changes might impact revenue.

This helps businesses prepare contingency plans.

## Scenario Comparison

The following comparison shows projected revenue under different business strategies:

- Baseline forecast
- Marketing campaign scenario
- Price increase scenario
- Demand decline scenario

This analysis helps identify which strategies generate the highest potential revenue.

### Monte Carlo Simulation 

Monte Carlo simulation generates thousands of possible sales outcomes by randomly varying demand within a specified range.

This approach helps estimate:

- Best case revenue
- Worst case revenue
- Expected revenue range

## Key Business Insights

Based on the analysis:

1. 

## Recommendations

Based on the findings:

- 